In [1]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [2]:
import time

In [3]:
import sys
from pathlib import Path

# add pharmalens/ root to Python path
sys.path.insert(0, str(Path.cwd().parent))
from agents.state import get_unprocessed_files, mark_file_processed

In [4]:
from agents.logger import get_logger
logger = get_logger("pharmalens.compiler")

In [5]:
load_dotenv()  # loads .env file — must be before genai.Client()

client = genai.Client()

FLASH_MODEL = "gemini-2.5-flash"

In [6]:
BASE_DIR = Path(os.getcwd()).parent 
WIKI_DIR      = BASE_DIR / "wiki"
REFERENCE_DIR = BASE_DIR / "reference"
PROMPTS_DIR   = BASE_DIR / "agents" / "prompts"
TEMPLATES_DIR = PROMPTS_DIR / "templates"     

SCHEMA        = (PROMPTS_DIR / "CLAUDE.md").read_text()

# load reference data once at module level
DRUGS = json.loads((REFERENCE_DIR / "drugs.json").read_text())
INDICATIONS = json.loads((REFERENCE_DIR / "indications.json").read_text())
COMPANIES = json.loads((REFERENCE_DIR / "companies.json").read_text())

In [7]:
BRAND_TO_INN = {}
for inn, data in DRUGS.items():
    for alias in data.get("aliases", []):
        BRAND_TO_INN[alias.lower()] = inn
    for brand in data.get("brand_names", []):
        BRAND_TO_INN[brand.lower()] = inn

ALIAS_TO_INDICATION = {}
for slug, data in INDICATIONS.items():
    for alias in data.get("aliases", []):
        ALIAS_TO_INDICATION[alias.lower()] = slug
    for code in data.get("icd10", []):
        ALIAS_TO_INDICATION[code.lower()] = slug

ALIAS_TO_COMPANY = {}
for slug, data in COMPANIES.items():
    for alias in data.get("aliases", []):
        ALIAS_TO_COMPANY[alias.lower()] = slug

In [8]:
def read_wiki_page(page_path: str) -> str:
    """Read an existing wiki page. Returns empty string if not found."""
    full_path = WIKI_DIR / page_path
    if full_path.exists():
        return full_path.read_text()
    return ""


def write_wiki_page(page_path: str, content: str) -> str:
    """Write content to a wiki page, creating parent directories as needed."""
    full_path = WIKI_DIR / page_path
    full_path.parent.mkdir(parents=True, exist_ok=True)
    full_path.write_text(content)
    return page_path

In [9]:
def load_prompt(doc_type: str) -> str:
    """Load the extraction prompt for a given document type.
    Raises ValueError if no prompt file exists — fail loud, not silent.
    """
    prompt_file = PROMPTS_DIR / f"compiler_{doc_type}.txt"
    if prompt_file.exists():
        return prompt_file.read_text()
    raise ValueError(
        f"No prompt file found for document type '{doc_type}'. "
        f"Expected: {PROMPTS_DIR}/compiler_{doc_type}.txt — "
        f"Either add the prompt file or update classify_document() in orchestrator.py"
    )


def load_page_template(page_type: str) -> str:
    """
    Load the formatting template for a specific wiki page type.
    Only injected in Step 3 calls — not in Step 1 extraction or index updates.
    Raises ValueError if template file is missing, same as load_prompt().
    """
    template_file = TEMPLATES_DIR / f"{page_type}.md"
    if template_file.exists():
        return template_file.read_text()
    raise ValueError(
        f"No template found for page type '{page_type}'. "
        f"Expected: {TEMPLATES_DIR}/{page_type}.md"
    )

In [10]:
def build_system_prompt() -> str:
    drug_summary = {
        k: {
            "inn": v["inn"],
            "brand_names": v["brand_names"],
            "company": v["company"],
            "indications": v["indications"]
        }
        for k, v in DRUGS.items()
    }
    indication_summary = {
        k: {"aliases": v.get("aliases", []), "icd10": v.get("icd10", [])}
        for k, v in INDICATIONS.items()
    }

    return f"""{SCHEMA}

Reference data (use for entity resolution — do NOT invent entity names):
DRUGS: {json.dumps(drug_summary, indent=2)}
INDICATIONS: {json.dumps(indication_summary, indent=2)}

Entity resolution rules:
- Always normalize brand names to INN using the DRUGS reference
- Always normalize indication aliases to indication slugs using INDICATIONS
- Never create wiki pages for entities not in the reference data
- If an entity is not in reference data, note it in extraction output but do not create pages
"""

In [11]:
def preprocess_ctgov(raw: str) -> str:
    """
    Parse CT.gov JSON and return only the fields defined in our extraction spec.
    Filters interventions to DRUG and BIOLOGICAL types only — ignores RADIATION,
    DEVICE, BEHAVIORAL etc which are not relevant for drug wiki pages.
    Falls back to raw text if JSON is malformed.
    """
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        return raw[:12000]

    relevant = {
        "nct_id":                   data.get("NCTId"),
        "brief_title":              data.get("BriefTitle"),
        "overall_status":           data.get("OverallStatus"),
        "phase":                    data.get("Phase"),
        "study_type":               data.get("StudyType"),
        "start_date":               data.get("StartDate"),
        "primary_completion_date":  data.get("PrimaryCompletionDate"),
        "completion_date":          data.get("CompletionDate"),
        "last_updated":             data.get("LastUpdatePostDate"),
        "lead_sponsor":             data.get("LeadSponsorName"),
        "lead_sponsor_class":       data.get("LeadSponsorClass"),
        "collaborator":             data.get("CollaboratorName"),
        "conditions":               data.get("Condition"),
        "brief_summary":            data.get("BriefSummary"),
        "enrollment_count":         data.get("EnrollmentCount"),
        "enrollment_type":          data.get("EnrollmentType"),
        "intervention_type":        data.get("InterventionType"),
        "intervention_name":        data.get("InterventionName"),
        "primary_outcome_measure":  data.get("PrimaryOutcomeMeasure"),
        "primary_outcome_timeframe":data.get("PrimaryOutcomeTimeFrame"),
        "has_results":              data.get("HasResults"),
    }

    # filter out non-drug intervention types
    if relevant.get("intervention_type") not in ("DRUG", "BIOLOGICAL"):
        relevant["intervention_type"] = None
        relevant["intervention_name"] = None


    # remove None values to keep output compact
    relevant = {k: v for k, v in relevant.items() if v is not None}

    return json.dumps(relevant, indent=2)

In [12]:
def preprocess_pubmed(raw: str) -> str:
    """
    Parse PubMed JSON and return only the fields defined in our extraction spec.
    Falls back to raw text if JSON is malformed.
    """
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        return raw[:12000]

    relevant = {
        "pmid": data.get("pmid"),
        "doi": data.get("doi"),
        "pubmed_date": data.get("pubmed_date"),
        "title": data.get("title"),
        "journal": data.get("journal"),
        "journal_abbr": data.get("journal_abbr"),
        "publication_types": data.get("publication_types", []),
        "abstract": data.get("abstract"),
        "mesh_major_topics": data.get("mesh_major_topics", []),
        "first_author": data.get("first_author"),
        "industry_sponsored": data.get("industry_sponsored"),
        "industry_note": data.get("industry_note"),
    }

    relevant = {k: v for k, v in relevant.items() if v is not None}

    return json.dumps(relevant, indent=2)

In [13]:
def preprocess_text(raw: str, max_chars: int) -> str:
    """
    Clean and intelligently truncate plain text documents.

    Removes blank lines to compact the text. If the document exceeds max_chars,
    keeps the first 70% and last 30% with a truncation notice in between.

    The 70/30 split is deliberate for earnings transcripts:
    - First 70%: CEO prepared remarks, CFO financial summary (highest signal)
    - Last 30%: analyst Q&A session often contains forward guidance and risk flags
    - Middle: less critical boilerplate, legal disclaimers, repetitive detail
    """
    # remove blank lines
    cleaned = "\n".join(
        line for line in raw.splitlines()
        if line.strip()
    )

    if len(cleaned) <= max_chars:
        return cleaned

    keep_start = int(max_chars * 0.70)
    keep_end = int(max_chars * 0.30)
    chars_dropped = len(cleaned) - max_chars

    return (
        cleaned[:keep_start]
        + f"\n\n[... {chars_dropped:,} characters truncated — middle section omitted ...]\n\n"
        + cleaned[-keep_end:]
    )

In [14]:
import re

# Boilerplate section headers that appear across all pharma company 8-Ks
# These are standardized because SEC filings follow industry conventions
_STRIP_MARKERS = [
    # Company about sections
    r"^About [A-Z][a-zA-Z\s]+$",          # "About AbbVie", "About Amgen", etc.
    # Boilerplate sections
    r"^Conference Call",
    r"^Non-GAAP Financial (?:Results|Measures)",
    r"^Forward-Looking Statements",
    r"^GAAP to Non-GAAP Reconciliation",
    r"^Reconciliation of GAAP",
]

# Exhibit markers — both naming conventions seen across companies
_PRESS_RELEASE_MARKERS = [
    "EARNINGS RELEASE",
    "PRESS RELEASE",
    "News Release",
    "EX-99.1",
]

def preprocess_edgar_8k(raw: str, max_chars: int = 8000) -> str:
    """
    8-K files are SEC wrappers around a press release exhibit (EX-99.1).
    
    Structure (consistent across all pharma companies):
      - YAML frontmatter (company, filing date, form type)    ← always keep
      - ## 8-K Main Body                                      ← strip entirely
          SEC boilerplate, Item 2.02 text, signature block
      - ## Exhibit ... - EX-99.1 / EARNINGS RELEASE           ← keep header
          Press release content                               ← keep
          "About {Company}" onward                            ← strip
    
    Returns: frontmatter + press release content, truncated to max_chars.
    """
    lines = raw.splitlines()

    # ── extract frontmatter ───────────────────────────────────────────────────
    # frontmatter is always between the first and second "---" markers
    frontmatter_end = 0
    dash_count = 0
    for i, line in enumerate(lines):
        if line.strip() == "---":
            dash_count += 1
            if dash_count == 2:
                frontmatter_end = i
                break
    frontmatter = "\n".join(lines[:frontmatter_end + 1])

    # ── find press release exhibit ────────────────────────────────────────────
    # look for the exhibit section header that contains the press release
    # both "## Exhibit ... - EARNINGS RELEASE" and "## Exhibit ... - EX-99.1"
    # followed shortly by "PRESS RELEASE" / "News Release" are valid markers
    exhibit_start = None
    for i, line in enumerate(lines):
        if line.startswith("## Exhibit"):
            # check if this exhibit is the press release
            # look ahead up to 10 lines for press release indicators
            lookahead = "\n".join(lines[i:i+10]).upper()
            if any(marker.upper() in lookahead for marker in _PRESS_RELEASE_MARKERS):
                exhibit_start = i
                break

    if exhibit_start is None:
        # no press release exhibit found — could be a non-earnings 8-K
        # (FDA decision, partnership, etc.) — skip stripping, return truncated full text
        logger.warning("preprocess_edgar_8k | No press release exhibit found — returning full text")
        return preprocess_text(raw, max_chars)

    exhibit_lines = lines[exhibit_start:]
    exhibit_text  = "\n".join(exhibit_lines)

    # ── strip boilerplate sections from within the press release ──────────────
    # find the earliest boilerplate marker and cut there
    # compiled once — these patterns are the same across all pharma companies
    strip_at = len(exhibit_text)
    for pattern in _STRIP_MARKERS:
        for match in re.finditer(pattern, exhibit_text, re.MULTILINE):
            if match.start() < strip_at:
                strip_at = match.start()

    exhibit_clean = exhibit_text[:strip_at].strip()

    # ── combine and truncate ──────────────────────────────────────────────────
    combined = frontmatter + "\n\n" + exhibit_clean
    return preprocess_text(combined, max_chars)

In [15]:
def preprocess_document(file_path: Path, doc_type: str) -> str:
    """
    Pre-process a raw file before passing to the LLM.

    JSON files (ctgov, pubmed): parse and extract only the fields the compiler
    needs — avoids passing hundreds of irrelevant fields to the LLM and wasting
    tokens on noise.

    Text files (edgar, genepool): clean whitespace and truncate intelligently,
    keeping both the start and end of long documents rather than blindly cutting.

    Returns a clean string ready for the Step 1 extraction prompt.
    """
    raw = file_path.read_text(errors="replace")

    if doc_type == "ctgov":
        return preprocess_ctgov(raw)
    elif doc_type == "pubmed":
        return preprocess_pubmed(raw)
    elif doc_type == "edgar_10q":
        # 10-Qs are long — allow more chars and keep start + end
        return preprocess_text(raw, max_chars=20000)
    elif doc_type == "edgar_8k":
        # 8-Ks are short — full content fits easily
        return preprocess_text(raw, max_chars=8000)
    elif doc_type == "genepool":
        # news articles are short
        return preprocess_text(raw, max_chars=8000)
    else:
        return preprocess_text(raw, max_chars=12000)

In [16]:
def preprocess_document(file_path: Path, doc_type: str) -> str:
    """
    Pre-process a raw file before passing to the LLM.

    JSON files (ctgov, pubmed): parse and extract only the fields the compiler
    needs — avoids passing hundreds of irrelevant fields to the LLM and wasting
    tokens on noise.

    Text/HTML files (edgar, genepool): clean and truncate intelligently.
    8-K files get the exhibit extractor before truncation.
    10-Q files get more chars since they contain XBRL + MD&A sections.

    Returns a clean string ready for the Step 1 extraction prompt.
    """
    raw = file_path.read_text(errors="replace")

    if doc_type == "ctgov":
        return preprocess_ctgov(raw)
    elif doc_type == "pubmed":
        return preprocess_pubmed(raw)
    elif doc_type == "edgar_8k":
        return preprocess_text(raw, max_chars=8000)
    elif doc_type == "edgar_10q":
        return preprocess_text(raw, max_chars=20000)
    elif doc_type == "genepool":
        return preprocess_text(raw, max_chars=8000)
    else:
        return preprocess_text(raw, max_chars=12000)

In [17]:
def get_company_from_path(path: Path) -> str | None:
    """Extract company slug from path for edgar and ctgov files."""
    parts = path.parts
    for source in ("edgar", "ctgov"):
        if source in parts:
            idx = list(parts).index(source)
            if idx + 1 < len(parts):
                return parts[idx + 1]
    return None


def get_drug_from_path(path: Path) -> str | None:
    """Extract drug INN from pubmed path."""
    parts = path.parts
    if "pubmed" in parts:
        idx = list(parts).index("pubmed")
        if idx + 1 < len(parts):
            return parts[idx + 1]
    return None


def classify_document(path: Path) -> str:
    """
    Infer document type from folder path — no LLM needed.
    Folder structure: raw/edgar/{company}/8k/ or raw/edgar/{company}/10Q/
    Returns: ctgov | edgar_8k | edgar_10q | genepool | pubmed | unknown
    """
    parts = path.parts

    if "ctgov" in parts:
        return "ctgov"
    elif "edgar" in parts:
        # classify by subfolder name, not filename
        parts_lower = [p.lower() for p in parts]
        if "10q" in parts_lower:
            return "edgar_10q"
        elif "8k" in parts_lower:
            return "edgar_8k"
        else:
            # fallback — unknown edgar subfolder
            logger.warning(f"classify_document | Unknown edgar subfolder for {path}")
            return "edgar_8k"
    elif "genepool" in parts:
        return "genepool"
    elif "pubmed" in parts:
        return "pubmed"
    else:
        return "unknown"

In [18]:
def create_cache(client, system_prompt: str, label: str, ttl: str = "3600s") -> str:
    cache = client.caches.create(
        model=FLASH_MODEL,
        config=types.CreateCachedContentConfig(
            system_instruction=system_prompt,
            ttl=ttl,
        )
    )
    print(f"  [CACHE] '{label}' cached: {cache.name} ({cache.usage_metadata.total_token_count} tokens)")
    return cache.name

In [19]:
def build_extraction_caches(
    client,
    unprocessed: list[Path],
) -> dict[str, str]:
    """
    Create one Gemini cache per doc type present in the unprocessed file list.
    Each cache contains: system prompt + extraction prompt for that doc type.

    Only caches doc types actually needed for this pipeline run — no wasted
    cache slots for doc types with no files queued.
    Skips any doc type whose prompt file is missing (ValueError from load_prompt).

    Returns dict keyed by doc_type string → cache name.
    """
    doc_types_needed = {classify_document(f) for f in unprocessed}
    extraction_caches = {}

    for doc_type in doc_types_needed:
        try:
            extraction_caches[doc_type] = create_cache(
                client,
                build_system_prompt() + load_prompt(doc_type),
                label=f"extraction_{doc_type}",
            )
        except ValueError as e:
            print(f"  [CACHE] Skipping extraction cache for '{doc_type}': {e}")

    return extraction_caches

In [20]:
def build_template_caches(client) -> dict[str, str]:
    """
    Create one Gemini cache per wiki page type.
    Each cache contains: system prompt + page formatting template.

    All 5 page types are always cached regardless of which pages will be
    written this run — templates are small and the full set is always needed
    since any doc type can trigger any combination of page writes.
    Skips any page type whose template file is missing (ValueError from load_page_template).

    Returns dict keyed by page_type string → cache name.
    """
    page_types = ["drug", "company", "trial", "event", "indication_hub"]
    template_caches = {}

    for page_type in page_types:
        try:
            template_caches[page_type] = create_cache(
                client,
                build_system_prompt() + load_page_template(page_type),
                label=f"template_{page_type}",
            )
        except ValueError as e:
            print(f"  [CACHE] Skipping template cache for '{page_type}': {e}")

    return template_caches

In [21]:
def compile_document_step1(file_path: Path, context: dict, raw_content: str, doc_type: str, system_prompt: str) -> dict:

    step1_prompt = f"""Document to process:
---
{raw_content}
---

File path: {file_path}
Document type: {doc_type}
Company context (from path): {context.get('company', 'unknown')}
Drug context (from path): {context.get('drug', 'unknown')}

Extract all relevant entities and signals. Return a JSON object:
{{
  "all_drugs_mentioned": [                      # everything in the document
    {{
        "name": "risankizumab",               # INN preferred, brand if INN unknown
        "brand_name": "Skyrizi",
        "is_tracked": false,          # ← True if name is in drugs.json
        "revenue_usd_m": 3425,                # null if not stated
        "revenue_growth_pct": 70.5,           # null if not stated
        "direction": "growing",               # growing | declining | flat | null
        "sentiment": "bullish",               # per-drug sentiment
        "commentary": "one sentence from management or results",
        "is_pipeline": false                  # true if not yet approved
    }}
],
  "companies_mentioned": ["novo-nordisk"],
  "indications_mentioned": ["glp1-obesity"],
  "trial_ids": ["NCT03819153"],
  "event_type": "fda_approval | trial_completion | trial_termination | earnings_signal | news | pubmed_result | null",
  "event_summary": "one sentence describing the key signal",
  "sentiment": "bullish | bearish | neutral | null",
  "sentiment_reasoning": "brief quote or paraphrase supporting sentiment",
  "key_facts": ["fact 1", "fact 2"],
  "event_date": "YYYY-MM-DD or null",
  "requires_new_event_page": true,
  "suggested_event_slug": "2026-04-02-semaglutide-ckd-fda-approval"
}}
"clinical_findings": {{
"study_design": "RCT | meta-analysis | systematic review | observational | cost-effectiveness | null",
"sample_size": "3533 patients | 39 RCTs | null",
"comparator": "placebo | standard of care | competitor drug | null",
"primary_outcome": "exact outcome measure as stated in abstract",
"primary_result": "exact result with numbers — e.g. HR 0.76, 95% CI 0.66-0.88, p<0.001",
"secondary_results": ["result 1 with numbers", "result 2 with numbers"],
"safety_note": "any adverse event finding or null",
"conclusions_verbatim": "copy the CONCLUSIONS section exactly as written",
"journal": "journal name",
"publication_year": "YYYY",
"industry_sponsored": true
}}
Return ONLY valid JSON with no markdown fences.
"""
    
    step1_response = client.models.generate_content(
        model=FLASH_MODEL,
        contents=step1_prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=0.1,
            response_mime_type="application/json"
        )
    )

    try:
        extracted = json.loads(step1_response.text)
    except json.JSONDecodeError:
        raise ValueError(
            f"Step 1 JSON parse failed for {file_path}. "
            f"Response was: {step1_response.text[:300]}"
        )

    return extracted

In [22]:
def collect_entity_pages(extracted: dict, pages_to_update: list) -> None:
    """
    Validate extracted entities against reference data.
    Appends page update entries for drugs, companies, and indications.
    Entities not found in reference files are logged and skipped.
    Modifies pages_to_update in place.
    """

    # enrich all_drugs_mentioned with is_tracked flag
    all_drugs = extracted.get("all_drugs_mentioned", [])
    for drug in all_drugs:
        drug["is_tracked"] = drug.get("name", "") in DRUGS
    extracted["all_drugs_mentioned"] = all_drugs

    # derive the tracked drug list from all_drugs_mentioned for page routing
    tracked_drug_names = [d["name"] for d in all_drugs if d.get("is_tracked")]

    entity_configs = [
        {
            "key":       "drugs_mentioned",
            "reference": DRUGS,
            "path_fn":   lambda e: f"drugs/{e}.md",
            "type":      "drug",
            "ref_name":  "reference/drugs.json",
            "entities":  tracked_drug_names,            # ← use derived list directly
        },
        {
            "key":       "companies_mentioned",
            "reference": COMPANIES,
            "path_fn":   lambda e: f"companies/{e}.md",
            "type":      "company",
            "ref_name":  "reference/companies.json",
        },
        {
            "key":       "indications_mentioned",
            "reference": INDICATIONS,
            "path_fn":   lambda e: f"indications/{e}/_index.md",
            "type":      "indication_hub",
            "ref_name":  "reference/indications.json",
        },
    ]

    for config in entity_configs:
        # use pre-derived list for drugs, extracted field for everything else
        entities = config.get("entities") or extracted.get(config["key"], [])
        for entity in entities:
            normalized = entity if entity in config["reference"] else entity.lower()
            if normalized in config["reference"]:
                page_path = config["path_fn"](normalized)
                pages_to_update.append({
                    "path":             page_path,
                    "type":             config["type"],
                    "entity":           normalized,
                    "current":          read_wiki_page(page_path),
                    "all_drugs_mentioned": all_drugs,   # carries is_tracked flag
                })
            else:
                logger.warning(f"STEP2 | SKIP ENTITY | '{entity}' not in {config['ref_name']}")

In [23]:
def collect_trial_and_event_pages(
    extracted: dict,
    pages_to_update: list,
    file_path: Path,
    context: dict, 
) -> bool:
    """
    Validate and accumulate trial and event page entries.
    Trials: one file per company, multiple NCT IDs accumulated onto one entry.
    Events: always a new page.
    Returns False if pages_to_update is still empty (caller should return early).
    """
    from agents.state import get_nct_action, mark_nct_processed

    for nct_id in extracted.get("trial_ids", []):
        if not (nct_id.startswith("NCT") and len(nct_id) == 11):
            logger.warning(f"STEP2 | SKIP ENTITY | Malformed trial ID: '{nct_id}'")
            continue

        primary_sponsor = next(
            (c for c in extracted.get("companies_mentioned", [])
             if c in COMPANIES or c.lower() in COMPANIES),
            None,
        )
        if primary_sponsor and primary_sponsor not in COMPANIES:
            primary_sponsor = primary_sponsor.lower()

        if not primary_sponsor:
            logger.warning(
                f"STEP2 | SKIP TRIAL | {nct_id} — no tracked company identified as sponsor"
            )
            continue
        
        if context.get("out_of_scope"):
            logger.info(
                f"STEP2 | SKIP TRIAL | {nct_id} — out of scope, company page only"
            )
            continue

        nct_action = get_nct_action(nct_id)
        mark_nct_processed(nct_id, primary_sponsor, extracted.get("event_date"))
        logger.debug(f"STEP2 | {nct_id} → action={nct_action}, sponsor={primary_sponsor}")

        trial_page_path = f"trials/{primary_sponsor}.md"
        existing = next((p for p in pages_to_update if p["path"] == trial_page_path), None)

        if existing is None:
            pages_to_update.append({
                "path":    trial_page_path,
                "type":    "trial",
                "entity":  primary_sponsor,
                "current": read_wiki_page(trial_page_path),
                "nct_ids": [{"nct_id": nct_id, "action": nct_action}],
            })
        else:
            existing["nct_ids"].append({"nct_id": nct_id, "action": nct_action})
            logger.debug(f"STEP2 | Accumulated {nct_id} onto {trial_page_path}")

    if extracted.get("requires_new_event_page") and extracted.get("suggested_event_slug"):
        event_path = f"events/{extracted['suggested_event_slug']}.md"
        pages_to_update.append({
            "path":    event_path,
            "type":    "event",
            "entity":  extracted["suggested_event_slug"],
            "current": "",
        })
        logger.debug(f"STEP2 | Queued event: {event_path}")

    if not pages_to_update:
        logger.info(f"STEP2 | NO PAGES | No tracked entities in {file_path.name}")
        return False

    return True

In [24]:
def build_trial_context(page: dict) -> str:
    """Build the trial-specific NCT instruction block for Step 3 prompts.
    Returns an empty string for all non-trial page types."""
    if page["type"] != "trial":
        return ""

    nct_details = page.get("nct_ids", [])
    new_ncts    = [n["nct_id"] for n in nct_details if n["action"] == "new"]
    update_ncts = [n["nct_id"] for n in nct_details if n["action"] == "update"]

    return f"""
Trial page instructions (determined by processing state — do not override):
- NCT IDs to ADD as new entries: {new_ncts if new_ncts else "none"}
- NCT IDs to UPDATE existing entries: {update_ncts if update_ncts else "none"}
- For ADD: create a new section under the correct status group (Active / Completed / Terminated)
  and add a new row to the summary table.
- For UPDATE: find the existing section by NCT ID and update status, result, and completion date.
  Do NOT add a duplicate section. Do NOT remove any other existing trial entries.
"""

In [25]:
def write_wiki_pages(
    pages_to_update: list,
    extracted: dict,
    file_path: Path,
    doc_type: str,
    template_caches: dict,
) -> list[str]:
    """
    Step 3 — write or update each wiki page identified in Step 2.
    One LLM call per page using the cached template for that page type.
    Returns the list of wiki page paths written.
    """
    updated_paths = []

    clinical_block = ""
    if extracted.get("clinical_findings"):
        cf = extracted["clinical_findings"]
        clinical_block = f"""
    Clinical findings to include verbatim in the Clinical evidence section:
    - Study design: {cf.get('study_design')}
    - Sample size: {cf.get('sample_size')}
    - Comparator: {cf.get('comparator')}
    - Primary outcome: {cf.get('primary_outcome')}
    - Primary result: {cf.get('primary_result')}
    - Secondary results: {cf.get('secondary_results', [])}
    - Safety: {cf.get('safety_note')}
    - Conclusions (verbatim): {cf.get('conclusions_verbatim')}
    - Journal: {cf.get('journal')}, {cf.get('publication_year')}
    - Industry sponsored: {cf.get('industry_sponsored')}

These numbers MUST appear in the Clinical evidence section exactly as stated.
Do not paraphrase the primary result or conclusions.
"""

    for page in pages_to_update:
        trial_context = build_trial_context(page)

        step3_prompt = f"""{trial_context}
{clinical_block}
Write the updated wiki page for: {page['path']}
Page type: {page['type']}
Entity: {page['entity']}

Extracted signals from the new document:
{json.dumps(extracted, indent=2)}

Current page content (empty if new page):
---
{page['current'] if page['current'] else '[NEW PAGE — write from scratch using the template above]'}
---

Source document: {file_path} (type: {doc_type})

Rules:
- Follow the page template exactly for this page type
- If the page exists: UPDATE it — preserve all existing content, only add or change what this document affects
- If new page: write complete page from scratch
- Timeline section is append-only — add new rows at the top, never delete existing rows
- End with a ## Sources section listing the raw file path
- For all_drugs_mentioned in the extracted signals: drugs with is_tracked=true have canonical wiki 
  pages — use [[drug_name]] link syntax for those only. Write is_tracked=false drugs as plain text.
- Write ONLY the markdown content — no preamble, no explanation
"""
        start = time.time()
        step3_response = client.models.generate_content(
            model=FLASH_MODEL,
            contents=step3_prompt,
            config=types.GenerateContentConfig(
                cached_content=template_caches[page["type"]],
                temperature=0.2,
            )
        )
        logger.info(f"STEP3 | API call took {time.time()-start:.1f}s for {page['type']}")

        write_wiki_page(page["path"], step3_response.text)
        updated_paths.append(page["path"])
        logger.info(f"STEP3 | WROTE | {page['path']}")

    return updated_paths

In [26]:
def compile_document(
    file_path: Path,
    context: dict,
    extraction_caches: dict | None = None,
    template_caches: dict | None = None,
) -> list[str]:
    """
    3-step compiler chain: extract → validate → write.

    Args:
        file_path:         Path to the raw source file.
        context:           Dict with keys: doc_type, company, drug.
        extraction_caches: Dict of doc_type → cache name (built by orchestrator).
                           If None, system prompt is built inline (notebook mode).
        template_caches:   Dict of page_type → cache name (built by orchestrator).
                           If None, system prompt is built inline (notebook mode).

    Returns:
        List of wiki page paths that were written.
    """
    from agents.state import update_index_py

    extraction_caches = extraction_caches or {}
    template_caches   = template_caches   or {}

    doc_type  = context["doc_type"]
    file_name = file_path.name

    logger.info(f"COMPILE | START | {file_name} | doc_type={doc_type} | "
                f"company={context.get('company')} | drug={context.get('drug')}")

    # pre-process document
    raw_content = preprocess_document(file_path, doc_type)

    # ── STEP 1: Extract ───────────────────────────────────────────────────────
    try:
        extracted = compile_document_step1(
            file_path,
            context,
            raw_content,
            doc_type,
            extraction_caches.get(doc_type),
        )
    except ValueError as e:
        logger.error(f"STEP1 | FAILED | {file_name}: {e}")
        raise

    if doc_type == "pubmed":
        extracted["requires_new_event_page"] = False
        extracted["suggested_event_slug"] = None
        extracted["event_type"] = None
        extracted["event_summary"] = None

    # ── STEP 2: Collect pages ─────────────────────────────────────────────────
    pages_to_update = []
    collect_entity_pages(extracted, pages_to_update)
    has_work = collect_trial_and_event_pages(extracted, pages_to_update, file_path, context)

    if not has_work:
        logger.info(f"COMPILE | DONE | {file_name} — no tracked entities, nothing written")
        return []

    # ── STEP 3: Write pages ───────────────────────────────────────────────────
    updated_paths = write_wiki_pages(
        pages_to_update, extracted, file_path, doc_type, template_caches
    )

    # ── update wiki navigation map (pure Python, no LLM) ──────────────────────
    if pages_to_update:
        update_index_py(pages_to_update)

    logger.info(
        f"COMPILE | DONE | {file_name} → "
        f"{len(updated_paths)} pages written: {updated_paths}"
    )
    return updated_paths

In [27]:
unprocessed = get_unprocessed_files()

In [28]:
doc_types_needed = {classify_document(f) for f in unprocessed}
doc_types_needed

{'ctgov', 'edgar_10q', 'edgar_8k', 'pubmed'}

In [29]:
extraction_caches = build_extraction_caches(client, unprocessed)

template_caches = build_template_caches(client)

  [CACHE] 'extraction_pubmed' cached: projects/665116874212/locations/us-central1/cachedContents/4242258804508459008 (4111 tokens)
  [CACHE] 'extraction_edgar_8k' cached: projects/665116874212/locations/us-central1/cachedContents/4327827197428498432 (4387 tokens)
  [CACHE] 'extraction_edgar_10q' cached: projects/665116874212/locations/us-central1/cachedContents/3229159994582630400 (4429 tokens)
  [CACHE] 'extraction_ctgov' cached: projects/665116874212/locations/us-central1/cachedContents/8984760318362124288 (4176 tokens)
  [CACHE] 'template_drug' cached: projects/665116874212/locations/us-central1/cachedContents/9110650001695965184 (4402 tokens)
  [CACHE] 'template_company' cached: projects/665116874212/locations/us-central1/cachedContents/4179208409725272064 (4159 tokens)
  [CACHE] 'template_trial' cached: projects/665116874212/locations/us-central1/cachedContents/6962432979440238592 (4186 tokens)
  [CACHE] 'template_event' cached: projects/665116874212/locations/us-central1/cachedCo

In [30]:
file_path = Path('/Users/mohitmahajan/Library/CloudStorage/OneDrive-Personal/Desktop/IUB/Projects/pharmalens/raw/edgar/abbvie/8k/2025-04-25-8K--25-000026.txt')


In [31]:
context = {
    "doc_type": 'edgar_8k',
    "company":  'abbvie',
    "drug":     None,
}

In [32]:
compile_document(file_path, context, extraction_caches, template_caches)

17:27:50 | INFO     | COMPILE | START | 2025-04-25-8K--25-000026.txt | doc_type=edgar_8k | company=abbvie | drug=None
17:28:15 | WARNING  | STEP2 | SKIP ENTITY | 'immunogen' not in reference/companies.json
17:28:29 | INFO     | STEP3 | API call took 13.8s for company
17:28:29 | INFO     | STEP3 | WROTE | companies/abbvie.md
17:28:42 | INFO     | STEP3 | API call took 12.4s for event
17:28:42 | INFO     | STEP3 | WROTE | events/2025-04-25-abbvie-q1-earnings.md
17:28:42 | INFO     | INDEX | added 2 new entries to index.md
17:28:42 | INFO     | COMPILE | DONE | 2025-04-25-8K--25-000026.txt → 2 pages written: ['companies/abbvie.md', 'events/2025-04-25-abbvie-q1-earnings.md']


['companies/abbvie.md', 'events/2025-04-25-abbvie-q1-earnings.md']

In [33]:
file_path = Path('/Users/mohitmahajan/Library/CloudStorage/OneDrive-Personal/Desktop/IUB/Projects/pharmalens/raw/ctgov/takeda/2026-04-07/NCT02747927.json')

In [34]:
context = {
    "doc_type": 'ctgov',
    "company":  'takeda',
    "drug":     None,
}

In [35]:
compile_document(file_path, context, extraction_caches, template_caches)

17:45:13 | INFO     | COMPILE | START | NCT02747927.json | doc_type=ctgov | company=takeda | drug=None
17:45:23 | WARNING  | STEP2 | SKIP ENTITY | 'dengue fever' not in reference/indications.json
17:45:39 | INFO     | STEP3 | API call took 15.3s for company
17:45:39 | INFO     | STEP3 | WROTE | companies/takeda.md
17:45:54 | INFO     | STEP3 | API call took 15.7s for trial
17:45:54 | INFO     | STEP3 | WROTE | trials/takeda.md
17:46:04 | INFO     | STEP3 | API call took 10.2s for event
17:46:04 | INFO     | STEP3 | WROTE | events/2024-06-28-takeda-tdv-trial-completion.md
17:46:04 | INFO     | INDEX | added 3 new entries to index.md
17:46:04 | INFO     | COMPILE | DONE | NCT02747927.json → 3 pages written: ['companies/takeda.md', 'trials/takeda.md', 'events/2024-06-28-takeda-tdv-trial-completion.md']


['companies/takeda.md',
 'trials/takeda.md',
 'events/2024-06-28-takeda-tdv-trial-completion.md']

In [36]:
context = {
    "doc_type": 'pubmed',
    "company":  None,
    "drug":     'semaglutide',
}

In [37]:
base_dir = '/Users/mohitmahajan/Library/CloudStorage/OneDrive-Personal/Desktop/IUB/Projects/pharmalens/raw/pubmed/semaglutide'
file_paths = list(Path(base_dir).rglob("*.json"))

In [39]:
compile_document(file_paths[1], context, extraction_caches, template_caches)

18:20:31 | INFO     | COMPILE | START | abstract-39903735.json | doc_type=pubmed | company=None | drug=semaglutide
18:20:43 | WARNING  | STEP2 | SKIP ENTITY | 'Metabolic dysfunction-associated steatohepatitis (MASH)' not in reference/indications.json
18:21:02 | INFO     | STEP3 | API call took 18.8s for drug
18:21:02 | INFO     | STEP3 | WROTE | drugs/tirzepatide.md
18:21:15 | INFO     | STEP3 | API call took 12.8s for drug
18:21:15 | INFO     | STEP3 | WROTE | drugs/semaglutide.md
18:21:42 | INFO     | STEP3 | API call took 27.4s for drug
18:21:42 | INFO     | STEP3 | WROTE | drugs/liraglutide.md
18:21:42 | INFO     | INDEX | added 2 new entries to index.md
18:21:42 | INFO     | COMPILE | DONE | abstract-39903735.json → 3 pages written: ['drugs/tirzepatide.md', 'drugs/semaglutide.md', 'drugs/liraglutide.md']


['drugs/tirzepatide.md', 'drugs/semaglutide.md', 'drugs/liraglutide.md']

In [40]:
compile_document(file_paths[2], context, extraction_caches, template_caches)

18:21:42 | INFO     | COMPILE | START | abstract-40069849.json | doc_type=pubmed | company=None | drug=semaglutide
18:22:07 | INFO     | STEP3 | API call took 14.4s for drug
18:22:07 | INFO     | STEP3 | WROTE | drugs/semaglutide.md
18:22:22 | INFO     | STEP3 | API call took 14.9s for company
18:22:22 | INFO     | STEP3 | WROTE | companies/novo-nordisk.md
18:22:37 | INFO     | STEP3 | API call took 14.8s for indication_hub
18:22:37 | INFO     | STEP3 | WROTE | indications/glp1-obesity/_index.md
18:22:51 | INFO     | STEP3 | API call took 14.3s for trial
18:22:51 | INFO     | STEP3 | WROTE | trials/novo-nordisk.md
18:22:51 | INFO     | INDEX | added 2 new entries to index.md
18:22:51 | INFO     | COMPILE | DONE | abstract-40069849.json → 4 pages written: ['drugs/semaglutide.md', 'companies/novo-nordisk.md', 'indications/glp1-obesity/_index.md', 'trials/novo-nordisk.md']


['drugs/semaglutide.md',
 'companies/novo-nordisk.md',
 'indications/glp1-obesity/_index.md',
 'trials/novo-nordisk.md']

In [41]:
compile_document(file_paths[3], context, extraction_caches, template_caches)

18:22:51 | INFO     | COMPILE | START | abstract-39882599.json | doc_type=pubmed | company=None | drug=semaglutide
18:22:58 | WARNING  | STEP2 | SKIP ENTITY | 'Novo Nordisk' not in reference/companies.json
18:22:58 | WARNING  | STEP2 | SKIP ENTITY | 'cardiovascular disease' not in reference/indications.json
18:22:58 | WARNING  | STEP2 | SKIP TRIAL | NCT03574597 — no tracked company identified as sponsor
18:23:23 | INFO     | STEP3 | API call took 25.1s for drug
18:23:23 | INFO     | STEP3 | WROTE | drugs/semaglutide.md
18:23:48 | INFO     | STEP3 | API call took 25.2s for indication_hub
18:23:48 | INFO     | STEP3 | WROTE | indications/glp1-obesity/_index.md
18:23:48 | INFO     | COMPILE | DONE | abstract-39882599.json → 2 pages written: ['drugs/semaglutide.md', 'indications/glp1-obesity/_index.md']


['drugs/semaglutide.md', 'indications/glp1-obesity/_index.md']

In [38]:
compile_document(file_paths[0], context, extraction_caches, template_caches)

18:19:45 | INFO     | COMPILE | START | abstract-39907981.json | doc_type=pubmed | company=None | drug=semaglutide
18:19:58 | WARNING  | STEP2 | SKIP ENTITY | 'heart-failure-with-preserved-ejection-fraction' not in reference/indications.json
18:19:58 | WARNING  | STEP2 | SKIP ENTITY | 'type-2-diabetes-mellitus' not in reference/indications.json
18:20:14 | INFO     | STEP3 | API call took 16.2s for drug
18:20:14 | INFO     | STEP3 | WROTE | drugs/semaglutide.md
18:20:24 | INFO     | STEP3 | API call took 9.7s for company
18:20:24 | INFO     | STEP3 | WROTE | companies/novo-nordisk.md
18:20:24 | INFO     | INDEX | added 2 new entries to index.md
18:20:24 | INFO     | COMPILE | DONE | abstract-39907981.json → 2 pages written: ['drugs/semaglutide.md', 'companies/novo-nordisk.md']


['drugs/semaglutide.md', 'companies/novo-nordisk.md']

In [ ]:
for file_path in file_paths:
    compile_document(file_path, context, extraction_caches, template_caches)
    mark_file_processed(file_path, doc_type, "success", company, drug)